# 04 · Guardrails: límite de llamadas y verificador de cifras

**Tarea de la práctica que cubre:** la tarea 2, "añadir un límite de llamadas a herramienta por invocación y un middleware propio que contraste las cifras con los datos XBRL, devolviendo el desajuste al modelo cuando no cuadren".

**Por qué hacen falta.** El diagnóstico del baseline (notebook 03) mostró que casi la mitad de los fallos son de cifra: el agente escribía en `cifra` un número que no correspondía (la variación entre ejercicios en lugar del valor del año). Un guardrail es una comprobación que se ejecuta *mientras* el agente trabaja y le corrige antes de dar la respuesta por buena.

Son tres protecciones, y el código completo cabe en un fichero corto, [`agente/guardrails.py`](../agente/guardrails.py):

1. **Límites de llamadas.** Ya vienen hechos en LangChain: `ToolCallLimitMiddleware` (herramientas) y `ModelCallLimitMiddleware` (modelo).
2. **Reintento con espera** (`ModelRetryMiddleware`), para los límites de peticiones del proveedor y sus errores transitorios.
3. **Verificador de cifras.** Es el middleware propio: cuando el modelo da su respuesta final, comprueba su `cifra` contra XBRL (y, en las comparaciones, también `cifra_anterior` y `variacion_pct`) y, si no cuadra, le devuelve el desajuste para que lo corrija. También exige la salida estructurada obligatoria: si el modelo termina con texto suelto, se le pide que use el esquema.

Este notebook explica cada una y la prueba en cuatro pasos. Los dos primeros no necesitan clave ni cuestan nada; los dos últimos usan el agente real (≈ 15 ¢).

In [1]:
import inspect
import os
import sys
from pathlib import Path

RAIZ = Path.cwd() if (Path.cwd() / "agente").exists() else Path.cwd().parent
sys.path.insert(0, str(RAIZ))

import pandas as pd

from agente import evaluadores, guardrails, trazas
from agente.agente import RespuestaFinanciera

print("Guardrails cargados:", [type(m).__name__ for m in guardrails.crear_guardrails()])

Guardrails cargados: ['ToolCallLimitMiddleware', 'VerificadorDeCifras']


## Guardrail 1 — El límite de llamadas

**Qué evita.** Un agente puede quedarse dando vueltas: llama a una herramienta, no le convence, vuelve a llamar… Cada vuelta cuesta dinero y tiempo. El límite corta esa espiral.

**Cómo funciona.** `ToolCallLimitMiddleware(run_limit=N)` cuenta las llamadas a herramienta de **una invocación** (una pregunta). Al pasar de N, las llamadas de más se bloquean con un mensaje y el modelo tiene que responder con lo que ya sabe. Elegimos `exit_behavior="continue"` y no `"end"`, porque así el agente sigue vivo y todavía puede devolver una respuesta estructurada en lugar de cortarse a la mitad.

**Hay dos límites, y no son lo mismo.** `ToolCallLimitMiddleware` cuenta llamadas a herramienta. `ModelCallLimitMiddleware` cuenta llamadas al **modelo**, y cubre el caso que el primero no ve: un modelo que da vueltas sin llamar a ninguna herramienta (algo que además el verificador podría provocar). Van los dos primeros de la lista, antes del verificador.

**Qué valor ponerle.** No a ojo: se mira cuántas llamadas hacía el baseline y se elige un valor **por encima del máximo observado**. El objetivo es cortar espirales, no cambiar el comportamiento normal: un límite que recortara preguntas legítimas sería un fallo nuevo.

In [2]:
BASELINE = pd.concat([pd.read_csv(RAIZ / f"resultados/eval_baseline_{n}.csv") for n in ("propio", "oficial", "huecos")])

display(BASELINE.groupby("familia").n_tools.agg(["mean", "max"]).round(1).rename(
    columns={"mean": "llamadas de media", "max": "máximo"}))
print(f"Límite elegido: {guardrails.LIMITE_LLAMADAS} llamadas por pregunta "
      f"(el baseline llegó como máximo a {int(BASELINE.n_tools.max())}).")

,llamadas de media,máximo
familia,,
comparativa,6.2,11
extractiva,3.5,5
numerica,3.6,8


Límite elegido: 12 llamadas por pregunta (el baseline llegó como máximo a 11).


## Guardrail 2 — El verificador de cifras

**La idea.** El agente tiene que devolver una respuesta estructurada (`RespuestaFinanciera`) con un campo `cifra`. Los datos XBRL son la fuente autorizada de cualquier cifra. Si lo que el agente afirma no coincide con XBRL, no se da por buena.

**Cómo funciona**, en cinco pasos:

1. El modelo escribe su respuesta final.
2. Justo después, un *hook* (`after_model`) recoge esa respuesta.
3. La función `revisar_cifra` la contrasta con XBRL y devuelve un aviso, o nada si todo cuadra.
4. Si hay aviso, se le devuelve al modelo como un mensaje que empieza por `VERIFICACIÓN:` y el agente vuelve a intentarlo.
5. Se corrige como mucho **2 veces**. Un guardrail no puede colgar al agente en un bucle.

**Las reglas de `revisar_cifra`:**

| Situación | Qué hace |
|---|---|
| No hay `cifra` | Nada que comprobar |
| `fuente='ninguna'` pero hay cifra | Avisa: si el dato no está en el corpus no se estima |
| `fuente='texto'` | Se deja pasar: sale del texto del informe y no hay hecho XBRL que comparar |
| Falta `ticker`, `ejercicio` o `concept_xbrl` | Avisa: sin ellos no se puede comprobar |
| El concepto no existe para esa empresa y ejercicio | Avisa, con el nombre del concepto |
| La cifra no coincide con el valor XBRL | Avisa con el valor correcto, y una pista si parece estar en millones |

**Las cifras de una comparación.** En una comparación entre dos ejercicios el agente recupera **dos** cifras y calcula una variación, pero `cifra` solo lleva una: la del ejercicio más reciente. Para no dejar sin comprobar las otras dos, el esquema tiene dos campos opcionales, `cifra_anterior` y `variacion_pct`, que solo se rellenan en las comparaciones. La función `revisar_comparacion` los contrasta con XBRL:

| Situación | Qué hace |
|---|---|
| Los dos campos vacíos | No es una comparación: no hace nada |
| `fuente` distinta de `xbrl` o `ambas`, o faltan `ticker`, `ejercicio` o `concept_xbrl` | No comprueba (eso ya lo avisa `revisar_cifra`) |
| El ejercicio anterior no existe en el corpus | Avisa: no se puede comparar |
| `cifra_anterior` no coincide con el valor XBRL del ejercicio anterior (0,1 %) | Avisa con el valor correcto |
| `variacion_pct` no coincide con la que sale de los dos valores XBRL (margen de 0,15 puntos) | Avisa con la variación correcta |

La variación se compara con la calculada a partir de XBRL, **no con los números que haya escrito el propio agente**. `revisar_respuesta` junta los avisos de `revisar_cifra` y de `revisar_comparacion`, y es lo que llama el verificador.

**Una regla más: la salida estructurada.** El enunciado exige que la respuesta sea siempre estructurada. Si el modelo termina con texto suelto o vacío, `structured_response` queda en `None` y la pregunta entera se pierde. Es el fallo intermitente que el baseline sufrió en `hx-05`. Para evitarlo, el mismo hook le pide al modelo que responda de nuevo con el esquema (con el mismo límite de 2 avisos).

**Tolerancia del 0,1 %.** Es más estricta que el 1 % del evaluador, a propósito: el agente recibe el valor exacto de `get_xbrl_fact`, así que copiarlo es fácil, y con un 0,1 % se distinguen dos hechos vecinos. Por ejemplo, el pasivo y el revenue de Microsoft en FY2024 están a solo un 0,59 %: con el 1 % un número se colaría por el otro.

**Convención sobre `cifra`.** El aviso le recuerda al modelo que en `cifra` va **el valor XBRL del ejercicio pedido**, y que si compara dos ejercicios, la variación va en `respuesta`. Es exactamente el error que encontró el diagnóstico.

Este es el código, completo:

In [3]:
print(inspect.getsource(guardrails.revisar_cifra))
print(inspect.getsource(guardrails.revisar_comparacion))
print(inspect.getsource(guardrails.revisar_respuesta))
print(inspect.getsource(guardrails.VerificadorDeCifras))

def revisar_cifra(r) -> str | None:
    """Aviso para el modelo si la cifra de `r` no cuadra con XBRL.

    Devuelve None cuando no hay nada que decir. `r` es una
    `RespuestaFinanciera`.
    """
    if r.cifra is None:
        return None            # sin cifra no hay nada que comprobar

    if r.fuente == "ninguna":
        return ("Has marcado fuente='ninguna' pero das una cifra. Si el dato "
                "no está en el corpus, deja `cifra` vacía y no lo estimes.")

    if r.fuente == "texto":
        return None            # sale del texto del informe: no hay hecho XBRL

    # fuente 'xbrl' o 'ambas': la cifra tiene que poder comprobarse
    if not (r.ticker and r.ejercicio and r.concept_xbrl):
        return ("Rellena `ticker`, `ejercicio` y `concept_xbrl` para poder "
                "verificar la cifra contra XBRL.")

    esperado = valor_xbrl(r.ticker, r.ejercicio, r.concept_xbrl)
    if esperado is None:
        return (f"{r.ticker} no reporta '{r.concept_xbrl}' en FY{r.e

## Prueba 1 — Las reglas, caso por caso (sin API)

Se construyen respuestas de ejemplo, cada una con un defecto distinto, y se comprueba que `revisar_cifra` avisa cuando debe y calla cuando no. La respuesta base es correcta: los pasivos de Microsoft en FY2024. Cada caso cambia una cosa. Es la forma más barata de comprobar un guardrail: sin modelo y sin gastar nada.

In [4]:
def resp(**cambios):
    """Una respuesta correcta (pasivos de Microsoft FY2024); cada caso cambia algo."""
    base = dict(respuesta="x", fuente="xbrl", ticker="MSFT", ejercicio=2024,
                concept_xbrl="Liabilities", cifra=243686000000.0)
    base.update(cambios)
    return RespuestaFinanciera(**base)


CASOS = [   # (caso, respuesta, ¿debe avisar?)
    ("Cifra correcta", resp(), False),
    ("Cifra en millones en vez de unidades", resp(cifra=243686.0), True),
    ("Otro hecho vecino (el revenue, a un 0,59 %)", resp(cifra=245122000000.0), True),
    ("Concepto que la empresa no reporta", resp(ticker="AMZN", ejercicio=2025, concept_xbrl="GrossProfit", cifra=1e11), True),
    ("Variación en vez del valor del ejercicio", resp(ejercicio=2025, concept_xbrl="ResearchAndDevelopmentExpense", cifra=2978000000.0), True),
    ("Falta el concept_xbrl", resp(concept_xbrl=None), True),
    ("fuente='ninguna' con una cifra inventada", resp(fuente="ninguna", cifra=5e9), True),
    ("fuente='ninguna' sin cifra (hueco bien respondido)", resp(fuente="ninguna", cifra=None), False),
    ("Cifra del texto del informe (fuente='texto')", resp(fuente="texto", concept_xbrl=None, cifra=244e9), False),
]

filas = []
for nombre, respuesta, debe_avisar in CASOS:
    aviso = guardrails.revisar_cifra(respuesta)
    filas.append({"caso": nombre, "avisa": aviso is not None,
                  "debía avisar": debe_avisar, "correcto": (aviso is not None) == debe_avisar})
display(pd.DataFrame(filas))
assert all(f["correcto"] for f in filas), "algún caso no se comporta como se esperaba"

print("Lo que recibiría el modelo en el caso de la variación:\n")
print(guardrails.revisar_cifra(CASOS[4][1]))

,caso,avisa,debía avisar,correcto
0,Cifra correcta,False,False,True
1,Cifra en millones en vez de unidades,True,True,True
2,"Otro hecho vecino (el revenue, a un 0,59 %)",True,True,True
3,Concepto que la empresa no reporta,True,True,True
4,Variación en vez del valor del ejercicio,True,True,True
5,Falta el concept_xbrl,True,True,True
6,fuente='ninguna' con una cifra inventada,True,True,True
7,fuente='ninguna' sin cifra (hueco bien respond...,False,False,True
8,Cifra del texto del informe (fuente='texto'),False,False,True


Lo que recibiría el modelo en el caso de la variación:

Tu cifra 2,978,000,000.00 no coincide con XBRL: ResearchAndDevelopmentExpense de MSFT en FY2025 es 32,488,000,000.00. En `cifra` va el valor XBRL del ejercicio pedido; si comparas dos ejercicios o das una variación, ponla en `respuesta`.


## Prueba 1b — Las cifras de una comparación (sin API)

Lo mismo que la prueba 1, ahora para `revisar_comparacion`. La respuesta base es una comparación **correcta**: el gasto en I+D de Microsoft pasó de 29.510 M en FY2024 a 32.488 M en FY2025, un +10,1 %. Cada caso cambia una cosa, con datos XBRL reales.

In [ ]:
BASE_COMPARACION = dict(respuesta="x", fuente="ambas", ticker="MSFT", ejercicio=2025,
                        concept_xbrl="ResearchAndDevelopmentExpense", cifra=32488000000.0,
                        cifra_anterior=29510000000.0, variacion_pct=10.1)


def comparacion(**cambios):
    """Una comparación correcta; cada caso cambia algo."""
    return RespuestaFinanciera(**{**BASE_COMPARACION, **cambios})


CASOS_COMPARACION = [   # (caso, respuesta, ¿debe avisar?)
    ("Comparación correcta (las tres cifras)", comparacion(), False),
    ("Variación redondeada a un entero (10 %)", comparacion(variacion_pct=10.0), False),
    ("cifra_anterior equivocada (la variación absoluta)", comparacion(cifra_anterior=2978000000.0), True),
    ("variacion_pct equivocada", comparacion(variacion_pct=30.0), True),
    ("Las dos equivocadas", comparacion(cifra_anterior=1.0, variacion_pct=99.0), True),
    ("Solo cifra_anterior, y bien", comparacion(variacion_pct=None), False),
    ("Ejercicio anterior fuera del corpus (FY2024 frente a FY2023)",
     comparacion(ejercicio=2024, cifra=29510000000.0, cifra_anterior=1.0, variacion_pct=5.0), True),
    ("No es una comparación (campos vacíos)", comparacion(cifra_anterior=None, variacion_pct=None), False),
    ("fuente='texto': no se comprueba", comparacion(fuente="texto", cifra_anterior=1.0), False),
]

filas = []
for nombre, respuesta, debe_avisar in CASOS_COMPARACION:
    aviso = guardrails.revisar_comparacion(respuesta)
    filas.append({"caso": nombre, "avisa": aviso is not None,
                  "debía avisar": debe_avisar, "correcto": (aviso is not None) == debe_avisar})
display(pd.DataFrame(filas))
assert all(f["correcto"] for f in filas), "algún caso no se comporta como se esperaba"

print("Lo que recibiría el modelo si se equivoca en la variación:\n")
print(guardrails.revisar_comparacion(comparacion(variacion_pct=30.0)))

## Prueba 2 — El bucle completo, con un modelo de mentira (sin API)

La prueba anterior comprueba la regla. Falta comprobar que el **mecanismo** funciona dentro de un agente de verdad: que el aviso llega al modelo y que este corrige.

Para no gastar nada se usa un modelo de mentira que tiene el guion escrito: su **primera** respuesta trae la cifra en millones (mal) y la **segunda** en unidades (bien). Si el verificador funciona, el agente debe terminar con la segunda, y entre las dos debe aparecer el mensaje de verificación.

Un detalle técnico: aquí se le pide la salida estructurada con `ToolStrategy` porque el modelo de mentira no sabe hacerlo de forma nativa. El agente real no lo necesita, pero la lógica del verificador es la misma.

In [5]:
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy
from langchain_core.language_models.fake_chat_models import GenericFakeChatModel
from langchain_core.messages import AIMessage


class ModeloDeMentira(GenericFakeChatModel):
    """Un modelo que responde lo que se le ha escrito, sin llamar a ninguna API."""
    def bind_tools(self, tools, **kwargs):
        return self


def respuesta_del_guion(cifra, n):
    argumentos = dict(respuesta="Pasivos de Microsoft", cifra=cifra, unidad="USD", ticker="MSFT",
                      ejercicio=2024, fuente="xbrl", concept_xbrl="Liabilities")
    return AIMessage(content="", tool_calls=[{"name": "RespuestaFinanciera", "args": argumentos, "id": f"r{n}"}])


guion = iter([respuesta_del_guion(243686.0, 1),            # primero, mal (en millones)
              respuesta_del_guion(243686000000.0, 2)])     # después, bien
agente_de_mentira = create_agent(
    model=ModeloDeMentira(messages=guion), tools=[],
    response_format=ToolStrategy(RespuestaFinanciera),
    middleware=[guardrails.VerificadorDeCifras()])

r = agente_de_mentira.invoke({"messages": [{"role": "user", "content": "¿Pasivos de Microsoft en FY2024?"}]})

for m in r["messages"]:
    print(f"{type(m).__name__:<13} {str(m.content)[:120]}")
print("\nCifra final:", r["structured_response"].cifra)
assert r["structured_response"].cifra == 243686000000.0, "el verificador no corrigió la cifra"

HumanMessage  ¿Pasivos de Microsoft en FY2024?
AIMessage     
ToolMessage   Returning structured response: respuesta='Pasivos de Microsoft' cifra=243686.0 unidad='USD' ticker='MSFT' ejercicio=2024
HumanMessage  VERIFICACIÓN: Tu cifra 243,686.00 no coincide con XBRL: Liabilities de MSFT en FY2024 es 243,686,000,000.00. Parece esta
AIMessage     
ToolMessage   Returning structured response: respuesta='Pasivos de Microsoft' cifra=243686000000.0 unidad='USD' ticker='MSFT' ejercici

Cifra final: 243686000000.0


## Prueba 2b — Si el modelo se olvida del formato (sin API)

El otro fallo que el guardrail evita: el modelo contesta con **texto suelto** en lugar de usar el esquema. En un agente con herramientas, LangChain da por terminada la conversación en cuanto el modelo responde sin llamar a ninguna herramienta, y `structured_response` se queda en `None`. Sin guardrail, la pregunta se pierde entera.

Se repite el experimento con un guion distinto: primero un texto suelto y luego la respuesta correcta. Se ejecuta dos veces, sin guardrail y con él. Al agente se le da una herramienta de relleno porque el agente real tiene herramientas, y eso es lo que activa esa regla de salida.

In [ ]:
from langchain.tools import tool


@tool
def herramienta_de_relleno() -> str:
    """No se usa: solo hace que el agente tenga herramientas, como el real."""
    return "ok"


def agente_con_guion(guion, middleware):
    return create_agent(model=ModeloDeMentira(messages=iter(guion)), tools=[herramienta_de_relleno],
                        response_format=ToolStrategy(RespuestaFinanciera), middleware=middleware)


guion = [AIMessage(content="Lo siento, no sé responder eso."),      # texto suelto: se olvida del esquema
         respuesta_del_guion(243686000000.0, 3)]                     # luego, bien
pregunta = {"messages": [{"role": "user", "content": "¿Pasivos de Microsoft en FY2024?"}]}

sin_guardrail = agente_con_guion(list(guion), []).invoke(pregunta)
con_guardrail = agente_con_guion(list(guion), [guardrails.VerificadorDeCifras()]).invoke(pregunta)

print("Sin guardrail: structured_response =", sin_guardrail["structured_response"])
print("Con guardrail: cifra =", con_guardrail["structured_response"].cifra)
assert sin_guardrail["structured_response"] is None
assert con_guardrail["structured_response"].cifra == 243686000000.0

## Prueba 2c — Una comparación con la variación equivocada (sin API)

El bucle completo, ahora con una comparación. El modelo de mentira responde primero con la variación mal (un 30 % en lugar del 10,1 %) y después bien. Si el verificador funciona con las cifras nuevas, el agente debe terminar con el 10,1 % y debe haber recibido un aviso. Reutiliza `agente_con_guion` de la prueba anterior.

In [ ]:
def respuesta_comparacion(n, **cambios):
    argumentos = {**BASE_COMPARACION, **cambios}
    return AIMessage(content="", tool_calls=[{"name": "RespuestaFinanciera", "args": argumentos, "id": f"c{n}"}])


guion = [respuesta_comparacion(1, variacion_pct=30.0),     # mal: la variación
         respuesta_comparacion(2)]                          # bien
pregunta = {"messages": [{"role": "user", "content": "¿Cuánto creció el gasto en I+D de Microsoft entre FY2024 y FY2025?"}]}

r = agente_con_guion(list(guion), [guardrails.VerificadorDeCifras()]).invoke(pregunta)
avisos = [m.content for m in r["messages"] if type(m).__name__ == "HumanMessage"][1:]

print("Aviso recibido:", avisos[0][:150] if avisos else "ninguno")
print("Variación final:", r["structured_response"].variacion_pct, "%")
assert r["structured_response"].variacion_pct == 10.1 and len(avisos) == 1, "el verificador no corrigió la variación"

## Prueba 3 — Con el agente real (necesita clave, ≈ 10-15 ¢)

Ahora el agente de verdad, con `crear_agente(middleware=crear_guardrails())`. Se le hacen cuatro preguntas elegidas por lo que enseñan:

| Pregunta | Por qué |
|---|---|
| `pr-c09` y `of-015` | Comparativas que **fallaban en el baseline** por poner la variación en `cifra`: aquí debería actuar el verificador |
| `pr-n10` | Los pasivos de Microsoft, el caso de los hechos vecinos: una numérica que debe pasar sin correcciones |
| `hx-01` | Un hueco (Amazon no reporta `GrossProfit`): debe responder `fuente='ninguna'` sin cifra y el verificador no debe intervenir |

Para cada una se muestra si el baseline acertó la cifra, cuántas correcciones hizo el verificador, el aviso que recibió el modelo y si al final la cifra cuadra. Si el agente termina sin respuesta estructurada, se muestra su último mensaje para entender por qué. Recuerda que el modelo no es determinista: si una repetición sale bien sin que el verificador haga falta, es normal.

In [6]:
if not os.environ.get("OPENROUTER_API_KEY"):
    from getpass import getpass
    os.environ["OPENROUTER_API_KEY"] = getpass("OPENROUTER_API_KEY: ")

from agente.agente import crear_agente

GOLDEN = {g["id"]: g for ruta in ("golden_set", "oficial_20", "huecos_humo")
          for g in evaluadores.cargar_golden(RAIZ / f"golden/{ruta}.jsonl")}
ANTES = BASELINE.set_index("id")["cifra_ok"].to_dict()          # ¿acertó la cifra el baseline?

agente = crear_agente(middleware=guardrails.crear_guardrails())

for ident in ["pr-c09", "of-015", "pr-n10", "hx-01"]:
    g = GOLDEN[ident]
    print(f"[{ident}] {g['pregunta'][:85]}")
    try:
        r = agente.invoke({"messages": [{"role": "user", "content": g["pregunta"]}]},
                          config={"configurable": {"thread_id": f"guardrails-{ident}"}})
    except Exception as e:
        print(f"   ERROR: {type(e).__name__}: {str(e)[:100]}\n")
        continue
    avisos = [m.content for m in r["messages"]
              if type(m).__name__ == "HumanMessage" and str(m.content).startswith(guardrails.MARCA)]
    s = r.get("structured_response")
    print(f"   baseline: cifra_ok = {ANTES.get(ident)}")
    print(f"   correcciones del verificador: {len(avisos)}")
    for aviso in avisos:
        print("      →", aviso[:170])
    if s is None:
        print("   SIN respuesta estructurada. Último mensaje del modelo:")
        print("     ", str(r["messages"][-1].content)[:200], "
")
        continue
    print(f"   final: cifra = {s.cifra} · esperada = {g.get('cifra_esperada')} · "
          f"cifra_ok = {evaluadores.cifra_coincide_xbrl(g, r)}\n")

[pr-c09] ¿Cuánto creció el gasto en I+D de Microsoft entre FY2024 y FY2025, y qué lo impulsó?


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

   baseline: cifra_ok = False
   correcciones del verificador: 1
      → VERIFICACIÓN: Tu cifra 2,978,000,000.00 no coincide con XBRL: ResearchAndDevelopmentExpense de MSFT en FY2025 es 32,488,000,000.00. En `cifra` va el valor XBRL del ejerci
   final: cifra = 32488000000.0 · esperada = 32488000000.0 · cifra_ok = True

[of-015] ¿Cuánto creció el revenue de NVIDIA entre FY2024 y FY2025, y a qué lo atribuye la dir
   baseline: cifra_ok = False
   correcciones del verificador: 1
      → VERIFICACIÓN: Tu cifra 69,575,000,000.00 no coincide con XBRL: Revenues de NVDA en FY2025 es 130,497,000,000.00. En `cifra` va el valor XBRL del ejercicio pedido; si comp
   final: cifra = 130497000000.0 · esperada = 130497000000.0 · cifra_ok = True

[pr-n10] ¿A cuánto ascendían los pasivos totales de Microsoft al cierre de su ejercicio fiscal
   baseline: cifra_ok = True
   correcciones del verificador: 0
   final: cifra = 243686000000.0 · esperada = 243686000000.0 · cifra_ok = True

[hx-01] ¿Cuál fue el

## Prueba 4 — El límite de llamadas en acción

Para ver el límite trabajar hay que ponerle un tope muy bajo. Se hace una pregunta comparativa, que normalmente necesita unas 6 llamadas, con `run_limit=1`: la primera llamada se ejecuta y las siguientes se bloquean. En la traza, las llamadas bloqueadas devuelven un mensaje de límite en lugar de un resultado, y aun así el agente tiene que terminar con una respuesta.

In [7]:
from langchain.agents.middleware import ToolCallLimitMiddleware

agente_limitado = crear_agente(middleware=[ToolCallLimitMiddleware(run_limit=1, exit_behavior="continue")])
g = GOLDEN["pr-c09"]
r = agente_limitado.invoke({"messages": [{"role": "user", "content": g["pregunta"]}]},
                           config={"configurable": {"thread_id": "limite-pr-c09"}})
trazas.pretty_trace(r)

  1. list_available()
       -> AAPL (Apple Inc.): ejercicios [2024, 2025], items ['1A', '7', '7A', '8'] AMZN (AMAZON COM INC): ejercicios [2024, 2025], items ['1A', '7', '7A', '8'] GOOGL (Alphabet Inc.): ejercicios [2024, 2025], it…
  2. get_xbrl_fact(concept='ResearchAndDevelopmentExpense', ticker='MSFT', fiscal_year=2024)
       -> Tool call limit exceeded. Do not make additional tool calls.
  3. RespuestaFinanciera(ejercicio=2025, ticker='MSFT', fuente='ninguna', concept_xbrl='ResearchAndDevelopmentExpense', respuesta='No es posible obtener los datos necesarios sobre el gasto en I+D de Microsoft para FY2024 y FY2025 ni los factores que lo impulsaron, ya que se ha alcanzado el límite de llamadas a las herramientas disponibles para consultar la información en el corpus.', cifra=None)
       -> Returning structured response: respuesta='No es posible obtener los datos necesarios sobre el gasto en I+D de Microsoft para FY2024 y FY2025 ni los factores que lo impulsaron, ya que se ha alca

## Lo que hacen los guardrails, y lo que no

**Hacen:**

- Cortan un agente atascado (límite) y le devuelven un desajuste comprobado contra XBRL antes de dar la respuesta por buena (verificador).
- Protegen especialmente lo que más falló en el baseline: la `cifra` de las comparativas.
- En las comparaciones comprueban las **tres cifras** (la más reciente, la anterior y la variación), no solo una.
- Exigen la salida estructurada: si el modelo se olvida del esquema, se le pide que lo use en lugar de perder la pregunta.
- Reintentan con espera (5 s, luego 10, 20 y 40) ante un límite de peticiones o un error transitorio del proveedor, en lugar de perder la pregunta. Importa el día 24: la cuenta actual limita a 20 peticiones por minuto.

**No hacen** (y conviene decirlo en la defensa):

- **Solo comprueban los campos estructurados**: `cifra` y, en las comparaciones, `cifra_anterior` y `variacion_pct`. No repasan los números escritos dentro de `respuesta`, y una cifra de `fuente='texto'` se deja pasar porque no hay hecho XBRL que comparar.
- **Cada corrección cuesta una llamada más al modelo**, así que suben el coste y la latencia. Lo que aportan y lo que cuestan se mide en el notebook 05.
- **Si tras 2 correcciones sigue mal, se deja pasar.** El evaluador lo marcará como fallo: el guardrail ayuda, no garantiza.

**Siguiente paso.** Este notebook prueba los guardrails aislados. En el **notebook 05** se conectan al agente que usan `responder()` y `evaluar()` y se mide su efecto sobre el golden, con su coste, frente al baseline.

In [8]:
print("NOTEBOOK 04 COMPLETADO — guardrails probados.")
print("Código: agente/guardrails.py")
print("Siguiente: notebook 05 (conectar los guardrails a responder() y medir el efecto frente al baseline)")

NOTEBOOK 04 COMPLETADO — guardrails probados.
Código: agente/guardrails.py
Siguiente: notebook 05 (conectar los guardrails a responder() y medir el efecto frente al baseline)
